# STT 평가 보고서

발표 연습 한 번의 STT(음성 인식 결과)를 대본의 평가 기준과 비교해 채점한 결과를 보여 줍니다.
이 노트북은 **보기만 합니다.** 채점 결과와 비슷한 말 목록은 로컬 DB(`outputs/rubrics.sqlite`)에서, 정답 라벨 비교와 반복 채점 지표는
`reports/results/*.json` 에서 읽으며 LLM 을 부르지 않습니다. 새로 만들려면 `experiments/` 의 스크립트를
`04_evaluate_takes.py` → `05_accuracy.py` → `06_stability.py` 순서로 실행합니다 (그 전에 대본 분석의 `01_build_rubrics.py` 가 필요합니다).

### 가상 STT 데이터

`data/virtual/stt/` 에 대본마다 연습 9번의 STT 가 슬라이드별로 있습니다. 대본을 바탕으로 만든 가상 데이터이고, 실제 음성 인식 결과처럼
문장부호가 거의 없고 간투사(`음`, `어`)·말 반복·띄어쓰기 오류가 섞여 있습니다.

| take | 시나리오 | 내용 |
|---|---|---|
| take1 | 충실 | 대본을 거의 그대로 말함 |
| take2 | 의역 | 뜻은 같지만 자기 말로 바꾸고, 문장을 합치거나 순서를 바꿈 |
| take3 | 누락 | 문장의 25~40% 를 빠뜨리고, 일부 수치를 `많이`, `꽤` 로 뭉뚱그림 |
| take4 | 실수 | 틀린 수치, 대본과 반대되는 말, 즉흥 발언, 틀렸다가 바로 고쳐 말한 경우가 섞임 |
| take5 | 혼합 | 의역 · 누락 · 틀린 수치 · 반대되는 말 · 즉흥 발언이 한 연습에 섞임 |
| take6 | 인식오류 | 대본대로 말했지만 음성 인식이 수치 4~6개를 잘못 적음 (발표자의 실제 실수 2개도 섞음) |
| take7 | 더듬기 | 간투사 · 말 끊김 · 고쳐 말하기가 많음. 음성 인식이 수치를 숫자로 적음 |
| take8 | 요약 | 슬라이드마다 1~3문장으로 요약. 수치를 어림해 말함 |
| take9 | 발음 | 음성 인식이 발음이 비슷한 단어 7~8개와 수치 1~2개를 잘못 적음 (`좌석` → `자석`) |

`data/virtual/stt_labels/` 에는 대본 문장마다 실제로 어떻게 말했는지 적은 **정답 라벨**이 있습니다 (5장에서만 읽습니다).

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from coverage_lab.datasets import load_take, take_files
from coverage_lab.results import read_result, result_table
from coverage_lab.store import connect, load_take_evaluations
from coverage_lab.stt_report import (
    conflict_log,
    segmentation_report,
    show_evaluation,
    similar_counts,
    similar_report,
    take_summary,
)
from script_coverage.version import FEATURE_VERSION

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

conn = connect()
takes = [load_take(p) for p in take_files()]
all_evaluations = {take.take_id: load_take_evaluations(conn, take) for take in takes}
print("기능 버전:", FEATURE_VERSION, "/ 채점한 연습:", len(all_evaluations), "/ 슬라이드 평가:", sum(len(v) for v in all_evaluations.values()))

기능 버전: 1.0 / 채점한 연습: 18 / 슬라이드 평가: 180


## 1. STT 문장 분리 점검

실제 음성 인식 결과는 문장부호가 거의 없어서 Kiwi 가 문장을 나눕니다. 여러 문장이 한 문장으로 붙으면 문장 정렬 구간과 근거 번호(T번호)가 거칠어집니다.
실제 Deepgram 결과를 넣었을 때도 이 표로 먼저 확인합니다.

| 열 | 읽는 법 |
|---|---|
| 평균 길이 · 최대 길이 · 120자 넘는 문장 | 긴 문장이 많으면 여러 문장이 붙은 것 — 음성 인식의 구두점 · 문단 옵션을 확인 |
| 정렬 구간 1 / 2 / 3문장, 3문장 구간 비율 | 대본 한 문장이 STT 여러 문장에 걸친 정도 |
| 정렬 안 된 대본 문장 | STT 에서 대본 단어를 하나도 못 찾은 문장 (빠뜨린 문장이거나 정렬 실패) |
| 남은 간투사 후보 | 뜻이 있을 수도 있어 지우지 않은 말(`그`, `저`, `뭐`, `이제` …)의 빈도 |

In [2]:
segmentation_report(conn, takes)

,take_id,STT 문장 수,평균 길이,최대 길이,120자 넘는 문장,정렬 구간 1 / 2 / 3문장,3문장 구간 비율,정렬 안 된 대본 문장,남은 간투사 후보
0,가상대본1_take1,46,43.2,85,0,37 / 3 / 1,0.02,0,"이제 1, 그러니까 1, 그 1"
1,가상대본1_take2,42,52.0,114,0,28 / 8 / 5,0.12,0,"그 3, 그러니까 2, 좀 1"
2,가상대본1_take3,32,39.3,71,0,32 / 3 / 2,0.05,4,"그 1, 그러니까 1"
3,가상대본1_take4,46,46.3,96,0,38 / 2 / 1,0.02,0,"그 3, 그러니까 1, 좀 1"
4,가상대본1_take5,41,47.4,111,0,29 / 9 / 2,0.05,1,"좀 2, 그 2, 그러니까 1"
5,가상대본1_take6,42,46.6,126,1,34 / 2 / 5,0.12,0,"그 2, 그러니까 1, 좀 1"
6,가상대본1_take7,45,42.8,87,0,37 / 2 / 2,0.05,0,"그 14, 저 4, 좀 1"
7,가상대본1_take8,21,44.5,90,0,32 / 5 / 2,0.05,2,그 2
8,가상대본1_take9,45,42.9,87,0,39 / 0 / 2,0.05,0,"그러니까 1, 그 1, 이제 1"
9,가상대본2_take1,52,40.7,73,0,42 / 1 / 1,0.02,0,"그 4, 이제 1"


## 2. 연습별 채점 결과

연습마다 세 가지 점수(0~1)와 Key Point 판정 분포입니다. 점수는 LLM 이 매기지 않고, 판정과 규칙 결과로 코드가 계산합니다.

| 점수 | 뜻 |
|---|---|
| content_coverage | Key Point 판정 점수(전달 1 · 일부 0.5 · 누락 0 · 모순 -0.5)의 중요도 가중 평균 |
| critical_fact_accuracy | 수치 · 이름 사실을 맞게 말한 정도 (맞게 1 · 어림해 0.5 · 빠짐 · 틀림 0). 발음이 비슷해 판단을 보류한 사실은 뺌 |
| script_fidelity | 대본 내용 형태소를 같은 순서로 말한 정도 — '전달했는가'가 아니라 '대본대로 말했는가' |
| similar_words · similar_numbers | 점수 비율에서 뺀(아직 확인하지 않은) 비슷한 말의 개수 |

In [3]:
take_summary(takes, all_evaluations)

,take_id,scenario,content_coverage,critical_fact_accuracy,script_fidelity,similar_words,similar_numbers,covered/partial/missing/contradicted,검증한 슬라이드,틀리게 말한 수치
0,가상대본1_take1,충실,1.000,1.000,0.918,0,0,39/0/0/0,0,0
1,가상대본1_take2,의역,0.956,0.961,0.537,0,0,34/5/0/0,0,0
2,가상대본1_take3,누락,0.551,0.505,0.683,0,0,14/13/12/0,0,0
3,가상대본1_take4,실수,0.791,0.922,0.889,0,0,33/0/0/6,6,4
4,가상대본1_take5,혼합,0.778,0.802,0.647,0,1,29/3/5/2,3,1
5,가상대본1_take6,인식오류,0.905,0.947,0.903,0,5,37/0/0/2,6,2
6,가상대본1_take7,더듬기,0.937,0.981,0.890,0,0,35/1/3/0,0,0
7,가상대본1_take8,요약,0.557,0.558,0.438,0,0,13/12/14/0,2,0
8,가상대본1_take9,발음,0.924,1.000,0.965,6,2,37/0/0/2,5,0
9,가상대본2_take1,충실,1.000,1.000,0.904,0,0,41/0/0/0,1,0


### 충돌 조건별 발동 기록

문장 판정은 LLM 이 하고, 코드가 그 판정과 규칙 결과가 어긋나는 문장을 찾아 교차 검증(LLM 한 번 더)에 보냅니다. 정답 라벨 없이 셀 수 있는 기록이라
실제 발표 데이터에서도 같은 표를 만들 수 있습니다.

| 열 | 뜻 |
|---|---|
| 발동 | 이 조건이 뜬 대본 문장 수 |
| 단독 | 이 조건 하나만 뜬 문장 수 — 이 조건이 없었으면 다시 판정하지 않았을 문장 |
| 판정 바뀜 | 교차 검증 뒤 판정이 1차와 달라진 문장 수 |

In [4]:
conflict_log(all_evaluations)

,발동,단독,판정 바뀜
충돌,,,
contradicted,42,22,20
low_confidence,3,2,2
judgment_incomplete,0,0,0
evidence_invalid,7,7,0
evidence_unrelated,1,0,0
fact_missing,0,0,0
sound_alike,21,0,21
speaker_changed,1,1,1
approximation,0,0,0


## 3. 비슷한 말 목록

음성 인식이 단어나 수치를 발음이 비슷한 다른 말로 적었는지, 발표자가 실제로 다르게 말했는지는 텍스트만으로 가릴 수 없습니다.
그래서 처음 채점에서는 **비슷한 말을 점수 비율에서 빼고(판단 보류) 위치와 함께 목록으로 남깁니다.** 리뷰 담당은 이 위치로
대본 문장 · STT 문장(T번호) · 원본 STT 구간을 찾아 사용자에게 확인합니다. `규칙 추정`과 `규칙 신호`는 판단을 돕는 참고 정보입니다.

In [5]:
similar_counts(conn)

,take_id,비슷한_단어,비슷한_수치
0,가상대본1_take5,0,1
1,가상대본1_take6,0,5
2,가상대본1_take9,6,2
3,가상대본2_take4,0,1
4,가상대본2_take5,0,1
5,가상대본2_take6,0,5
6,가상대본2_take9,7,1


음성 인식이 발음이 비슷한 단어를 잘못 적은 연습(`가상대본1_take9`)의 목록입니다.

In [6]:
similar_report(conn, takes, "가상대본1_take9")

,slide,종류,대본,STT,대본 위치,STT 위치,원본 STT 위치,Key Point,규칙 추정,규칙 신호
0,2,단어,좌석,자석,S0 [9:11],T0 [11:13],[11:13] '자석',KP1,asr_error,대본 단어가 이 자리에서 빠지고 발음이 비슷한 다른 말이 나옴
1,2,수치,약 18퍼센트,약 팔 퍼센트,S2 [125:132],T2 [121:128],[121:128] '약 팔 퍼센트',KP3,asr_error,음절 하나('십')가 빠짐 → 음성 인식 오류일 수 있음
2,3,수치,30분,삼 분,S2 [115:118],T2 [110:113],[110:113] '삼 분',KP3,asr_error,음절 하나('십')가 빠짐 → 음성 인식 오류일 수 있음
3,3,단어,센서,센터,S4 [219:221],T4 [212:214],[214:216] '센터',KP5,asr_error,대본 단어가 이 자리에서 빠지고 발음이 비슷한 다른 말이 나옴
4,5,단어,이용률,이용료,S4 [171:174],T4 [170:173],[170:173] '이용료',"KP5, KP6",asr_error,대본 단어가 이 자리에서 빠지고 발음이 비슷한 다른 말이 나옴
5,6,단어,노쇼,노소,S3 [143:145],T3 [148:150],[148:150] '노소',KP4,asr_error,대본 단어가 이 자리에서 빠지고 발음이 비슷한 다른 말이 나옴
6,8,단어,리포트,리포터,S2 [93:96],T2 [87:90],[87:90] '리포터',KP3,asr_error,대본 단어가 이 자리에서 빠지고 발음이 비슷한 다른 말이 나옴
7,9,단어,대전,대천,S1 [33:35],T1 [36:38],[36:38] '대천',KP1,asr_error,대본 단어가 이 자리에서 빠지고 발음이 비슷한 다른 말이 나옴


## 4. 슬라이드 상세

대본 문장별 1차 판정(LLM) · 최종 판정 · 근거 · 단어 비율 · 충돌 이유, said 가 아닌 문장의 전달 단위 판정, 문장 판정을 모은 Key Point 판정,
사실 검증 결과, 비슷한 말을 차례로 보여 줍니다.

잘못 말한 수치와 말을 고쳐 말한 경우가 섞인 슬라이드입니다.

In [7]:
show_evaluation(all_evaluations, "가상대본1_take4", 6)

[가상대본1_take4 / 슬라이드 6] 점수: {'content_coverage': 0.7, 'critical_fact_accuracy': 0.92, 'script_fidelity': 0.963, 'similar_words': 0, 'similar_numbers': 0}
  [T0] 이천이십오 년 삼 월부터 8주 동안 제휴 도서관 3곳에서 시범 운영을 했습니다
  [T1] 참여한 이용자는 1860명이었고 빈자리를 찾는 데 걸린 시간은 평균 23분에서 십삼 분으로 약 이십사 퍼센트 아니 사십이 퍼센트 줄었습니다.
  [T2] 알림을 받고 10분 안에 자리를 잡은 비율은 67퍼센트였습니다
  [T3] 노쇼 비율도 18퍼센트에서 칠 퍼센트로 낮아졌고요 그 시범 운영 뒤에 설문에서 이용자의 85퍼센트 이상이 계속 쓰고 싶다고 답했습니다
대본 문장별 판정


,문장,역할,1차,최종,검증,근거,단어 비율,충돌,이유
0,S0,detail,said,said,False,[0],1.000,,"2025년 3월, 8주 동안, 제휴 도서관 3곳에서 시범 운영했다는 내용이 모두 전달되었습니다."
1,S1,claim,said,said,False,[1],1.000,,"참여자 수와 시간 단축 내용은 전달되었으며, 24퍼센트라고 잘못 말한 뒤 42퍼센트로 바로 정정했습니다."
2,S2,evidence,contradicted,contradicted,True,[2],0.833,contradicted,"[검증] 알림을 받고 10분 안에 자리를 잡은 비율이라는 내용은 전달했지만, 대본의 76%가 아니라 67%라고 말해 수치가 다릅니다."
3,S3,evidence,said,said,False,[3],1.000,,노쇼 비율이 18%에서 7%로 낮아졌다는 내용이 전달되었습니다.
4,S4,evidence,said,said,False,[3],1.000,,시범 운영 뒤 설문에서 이용자의 85% 이상이 계속 쓰고 싶다고 답했다는 내용이 전달되었습니다.


전달 단위 판정 (said 가 아닌 문장 — 문장 판정은 이 판정을 모은 것)


,문장,단위,내용,1차,최종
0,S2,S2-U1,알림을 받고 10분 안에 자리를 잡은 비율은 76%였음,contradicted,contradicted


Key Point 판정 (근거 문장 판정을 모은 것)


,id,importance,근거 문장,1차,최종,검증,내용
0,KP1,normal,[S0],covered,covered,False,2025년 3월부터 8주 동안 제휴 도서관 3곳에서 시범 운영을 했다.
1,KP2,critical,[S1],covered,covered,False,빈자리를 찾는 데 걸린 시간이 평균 23분에서 13분으로 약 42퍼센트 줄었다.
2,KP3,high,[S2],contradicted,contradicted,True,알림을 받고 10분 안에 자리를 잡은 비율이 76%이다.
3,KP4,high,[S3],covered,covered,False,노쇼 비율이 18%에서 7%로 낮아졌다.
4,KP5,high,[S4],covered,covered,False,시범 운영 뒤 설문에서 이용자의 85퍼센트 이상이 계속 쓰고 싶다고 답했다.


,id,type,대본,결과,STT,원인,note
0,CF1,date,2025년 3월,matched,이천이십오 년 삼 월,,
1,CF2,duration,8주 동안,matched,8주 동안,,
2,CF3,quantity,3곳,matched,3곳,,
3,CF4,quantity,"1,860명",matched,1860명,,
4,CF5,duration,평균 23분,matched,평균 23분,,
5,CF6,duration,13분,matched,십삼 분,,
6,CF7,percentage,약 42퍼센트,matched,사십이 퍼센트,,한정어 차이
7,CF8,duration,10분,matched,10분,,
8,CF9,percentage,76%,mismatched,67퍼센트,speaker_error,발음이 비슷하지 않거나 음절 순서만 바뀜 — 발표자가 다르게 말함
9,CF10,percentage,18%,matched,18퍼센트,,


음성 인식이 수치 두 개를 발음이 비슷한 다른 값으로 적은 슬라이드입니다 (`천이백` → `천백`, `이십사억` → `십사억`). 이 수치는 비율에서 빠집니다.

In [8]:
show_evaluation(all_evaluations, "가상대본1_take6", 7)

[가상대본1_take6 / 슬라이드 7] 점수: {'content_coverage': 1.0, 'critical_fact_accuracy': 1.0, 'script_fidelity': 0.968, 'similar_words': 0, 'similar_numbers': 2}
  [T0] 국내 공공도서관은 약 천백 곳이고 대학 도서관은 약 사백삼십 곳입니다
  [T1] 이 중에서 열람실 좌석 예약 시스템을 이미 쓰고 있는 곳을 초기 대상으로 보면 약 900곳이고 연간 시장 규모는 약 14억 원에서 36억 원으로 추산했습니다
  [T2] 그 처음에는 이용자가 제일 많은 서울이랑 대전의 대학 도서관부터 시작하겠습니다
대본 문장별 판정


,문장,역할,1차,최종,검증,근거,단어 비율,충돌,이유
0,S0,evidence,contradicted,said,True,[0],0.857,"contradicted, sound_alike","[검증] 약 1,200곳과 약 430곳 모두 STT에서 각각 발음이 비슷한 표현과 함께 전달되었으므로, 발음이 비슷한 수치 차이는 제외하..."
1,S1,claim,contradicted,said,True,[1],0.941,"contradicted, sound_alike","[검증] 약 900곳, 연간 시장 규모 약 24억 원에서 36억 원이라는 두 내용이 모두 전달되었습니다. 약 24억 원과 약 14억 원의..."
2,S2,detail,said,said,False,[2],0.889,,이용자가 가장 많은 서울과 대전의 대학 도서관부터 시작한다는 내용이 모두 전달되었다.


전달 단위 판정 (said 가 아닌 문장 — 문장 판정은 이 판정을 모은 것)


,문장,단위,내용,1차,최종
0,S0,S0-U1,"국내 공공도서관은 약 1,200곳임",contradicted,said
1,S0,S0-U2,대학 도서관은 약 430곳임,said,said
2,S1,S1-U1,열람실 좌석 예약 시스템을 이미 쓰는 약 900곳을 초기 대상으로 봄,said,said
3,S1,S1-U2,연간 시장 규모는 약 24억 원에서 36억 원으로 추산함,contradicted,said


Key Point 판정 (근거 문장 판정을 모은 것)


,id,importance,근거 문장,1차,최종,검증,내용
0,KP1,high,[S0],contradicted,covered,True,"국내 공공도서관은 약 1,200곳이고 대학 도서관은 약 430곳이다."
1,KP2,critical,[S1],contradicted,covered,True,열람실 좌석 예약 시스템을 이미 쓰는 초기 대상은 약 900곳이며 연간 시장 규모는 약 24억 원에서 36억 원으로 추산했다.
2,KP3,normal,[S2],covered,covered,False,초기에는 이용자가 가장 많은 서울과 대전의 대학 도서관부터 시작한다.


,id,type,대본,결과,STT,원인,note
0,CF1,quantity,"약 1,200곳",sound_alike,약 천백 곳,,발음이 비슷한 다른 말로 나옴 — 판단 보류 (비율에서 뺌)
1,CF2,quantity,약 430곳,matched,약 사백삼십 곳,,
2,CF3,quantity,약 900곳,matched,약 900곳,,
3,CF4,money,약 24억 원,sound_alike,약 14억 원,,발음이 비슷한 다른 말로 나옴 — 판단 보류 (비율에서 뺌)
4,CF5,money,36억 원,matched,36억 원,,
5,CF6,proper_noun,서울,matched,서울,,
6,CF7,proper_noun,대전,matched,대전,,


비슷한 말 (비율에서 뺌 — 리뷰 agent 가 사용자에게 확인)


,id,종류,대본,STT,대본 위치,STT 위치,Key Point,규칙 추정
0,R1,number,"약 1,200곳",약 천백 곳,"S0 [10, 18]","T0 [10, 16]",KP1,asr_error
1,R2,number,약 24억 원,약 14억 원,"S1 [100, 107]","T1 [102, 109]",KP2,asr_error


핵심 주장 문장을 빠뜨린 슬라이드입니다.

In [9]:
show_evaluation(all_evaluations, "가상대본1_take3", 7)

[가상대본1_take3 / 슬라이드 7] 점수: {'content_coverage': 0.333, 'critical_fact_accuracy': 0.267, 'script_fidelity': 0.6, 'similar_words': 0, 'similar_numbers': 0}
  [T0] 국내 공공도서관은 약 천이백 곳이고요
  [T1] 대학 도서관도 상당히 많습니다
  [T2] 그러니까 처음에는 이용자가 가장 많은 서울이랑 대전의 대학 도서관부터 시작하겠습니다
대본 문장별 판정


,문장,역할,1차,최종,검증,근거,단어 비율,충돌,이유
0,S0,evidence,partial,partial,False,"[0, 1]",0.857,,"국내 공공도서관 약 1,200곳은 말했지만 대학 도서관 수는 구체적인 약 430곳이 아니라 많다고만 말했습니다."
1,S1,claim,missing,missing,False,[],0.000,,열람실 좌석 예약 시스템을 이미 쓰는 약 900곳과 연간 시장 규모 약 24억 원에서 36억 원이라는 내용이 나오지 않았습니다.
2,S2,detail,said,said,False,[2],1.000,,이용자가 가장 많은 서울과 대전의 대학 도서관부터 시작하겠다고 말했습니다.


전달 단위 판정 (said 가 아닌 문장 — 문장 판정은 이 판정을 모은 것)


,문장,단위,내용,1차,최종
0,S0,S0-U1,"국내 공공도서관은 약 1,200곳임",said,said
1,S0,S0-U2,대학 도서관은 약 430곳임,vague,vague
2,S1,S1-U1,열람실 좌석 예약 시스템을 이미 쓰는 약 900곳을 초기 대상으로 봄,missing,missing
3,S1,S1-U2,연간 시장 규모는 약 24억 원에서 36억 원으로 추산함,missing,missing


Key Point 판정 (근거 문장 판정을 모은 것)


,id,importance,근거 문장,1차,최종,검증,내용
0,KP1,high,[S0],partial,partial,False,"국내 공공도서관은 약 1,200곳이고 대학 도서관은 약 430곳이다."
1,KP2,critical,[S1],missing,missing,False,열람실 좌석 예약 시스템을 이미 쓰는 초기 대상은 약 900곳이며 연간 시장 규모는 약 24억 원에서 36억 원으로 추산했다.
2,KP3,normal,[S2],covered,covered,False,초기에는 이용자가 가장 많은 서울과 대전의 대학 도서관부터 시작한다.


,id,type,대본,결과,STT,원인,note
0,CF1,quantity,"약 1,200곳",matched,약 천이백 곳,,
1,CF2,quantity,약 430곳,missing,NaN,,
2,CF3,quantity,약 900곳,missing,NaN,,
3,CF4,money,약 24억 원,missing,NaN,,
4,CF5,money,36억 원,missing,NaN,,
5,CF6,proper_noun,서울,matched,서울,,
6,CF7,proper_noun,대전,matched,대전,,


## 5. 정답 라벨과 비교 (성능 측정)

정답 라벨(대본 문장 단위)로 문장 · Key Point · 사실의 정답을 만들고 판정을 비교한 결과입니다
(`experiments/05_accuracy.py` 가 `reports/results/stt_accuracy.json` 에 남깁니다).

- **문장 정답**: 라벨 그대로 (verbatim · paraphrased → said, partial, missing, contradicted)
- **Key Point 정답**: 근거 문장 중 하나라도 contradicted 면 contradicted, 모두 전달이면 covered, 모두 missing 이면 missing, 나머지는 partial
- **사실 정답**: 한 번이라도 맞게 말했으면 matched, 음성 인식이 잘못 적었으면 asr, 어림해 말했으면 approximate, 틀린 값이면 mismatched, 아니면 missing

| 방법 | 내용 |
|---|---|
| 규칙만 | LLM 없이 규칙 결과로만 문장 판정 (기준선) |
| LLM 1차 | LLM 의 문장 판정 그대로 |
| 최종 | 충돌 검사와 교차 검증을 거친 판정 |

'심각한 오판'은 전달(covered / said)과 누락 · 모순을 뒤바꾼 경우입니다. 정답 라벨은 문장 단위라서 Key Point 정답은 문장 라벨로 계산한 근사값입니다.
이 값은 LLM 응답에 따라 달라지는, 이 실행의 결과입니다.

In [10]:
accuracy = read_result("stt_accuracy")
print("모델:", accuracy["model"], "/ 기능 버전:", accuracy["feature_version"], "/ 개수:", accuracy["counts"])
print("대본 문장 판정")
display(result_table("stt_accuracy", "sentence_accuracy"))
print("Key Point 판정")
display(result_table("stt_accuracy", "key_point_accuracy"))
print("시나리오별 최종 정확도 — 문장:", accuracy["sentence_accuracy_by_scenario"])
print("시나리오별 최종 정확도 — Key Point:", accuracy["key_point_accuracy_by_scenario"])
print("핵심 사실 정확도 (판단 보류 제외):", accuracy["fact_accuracy"])
print("슬라이드 Content Coverage 평균 절대 오차 (0~1):", accuracy["slide_coverage_error"])

모델: openai/gpt-5.6-luna / 기능 버전: 1.0 / 개수: {'sentences': 765, 'key_points': 720, 'facts': 738, 'held_facts': 18}
대본 문장 판정


,정확도,심각한 오판 비율
규칙만,0.848,0.020
LLM 1차,0.970,0.024
최종,0.992,0.004


Key Point 판정


,정확도,심각한 오판 비율
규칙만,0.846,0.019
LLM 1차,0.967,0.026
최종,0.992,0.004


시나리오별 최종 정확도 — 문장: {'누락': 1.0, '더듬기': 1.0, '발음': 1.0, '실수': 0.988, '요약': 0.965, '의역': 1.0, '인식오류': 1.0, '충실': 1.0, '혼합': 0.976}
시나리오별 최종 정확도 — Key Point: {'누락': 1.0, '더듬기': 1.0, '발음': 1.0, '실수': 0.988, '요약': 0.963, '의역': 1.0, '인식오류': 1.0, '충실': 1.0, '혼합': 0.975}
핵심 사실 정확도 (판단 보류 제외): 0.999
슬라이드 Content Coverage 평균 절대 오차 (0~1): {'LLM 1차': 0.044, '최종': 0.011}


### 비슷한 말 보류 성능

라벨의 인식 오류(영문 이름 제외)를 비슷한 말로 보류했는지(보류율), 보류한 것 중 실제 인식 오류와 실제 발표자 실수가 얼마나 되는지, 규칙 추정이 실제 원인을 맞힌 비율입니다.
발표자 실수는 사용자에게 확인해 가릴 몫입니다.

In [11]:
pd.Series(accuracy["similar"], name="값").to_frame()

,값
인식 오류를 보류한 비율,1.000
보류한 것 중 실제 인식 오류,0.897
보류한 것 중 실제 발표자 실수 (사용자 확인으로 가릴 것),0.103
규칙 추정 정확도,0.897


### 충돌 조건 점검

충돌 조건이 LLM 1차 판정의 실수를 제대로 가리키는지 정답 라벨로 봅니다.

| 열 | 뜻 |
|---|---|
| 발동 · 단독 | 2장 발동 기록과 같음 |
| 1차 오답 | 이 조건이 뜬 문장 중 LLM 1차 판정이 정답과 다른 수 — 조건이 실제 실수를 가리킨 수 |
| 단독 1차 오답 | 이 조건 하나만 떠서 잡은 1차 오답 — 이 조건이 없었으면 놓쳤을 실수 |
| 바로잡음 · 망침 | 교차 검증이 1차 오답을 정답으로 바꾼 수 · 1차 정답을 오답으로 바꾼 수 |

In [12]:
result_table("stt_accuracy", "conflict_check")

,발동,단독,1차 오답,단독 1차 오답,바로잡음,망침
contradicted,42,22,17,0,17,3
low_confidence,3,2,2,1,2,0
judgment_incomplete,0,0,0,0,0,0
evidence_invalid,7,7,0,0,0,0
evidence_unrelated,1,0,0,0,0,0
fact_missing,0,0,0,0,0,0
sound_alike,21,0,18,0,18,3
speaker_changed,1,1,1,1,1,0
approximation,0,0,0,0,0,0
partial_complete,0,0,0,0,0,0


### 사용자 확인 뒤 정확도

리뷰 담당이 비슷한 말을 사용자에게 확인한 뒤의 성능입니다. **정답 라벨로 사용자 답을 대신합니다.**
라벨이 인식 오류로 적은 항목은 '대본대로 말함', 발표자가 바꿔 말한 항목은 'STT 대로 말함'으로 답했다고 보고 점수를 다시 계산합니다 (LLM 호출 없음).
처음 채점에서 보류해 정답과 달랐던 발표자 실수가 바로잡히는지, 확인 때문에 다른 판정이 틀어지지 않는지 봅니다.

In [13]:
confirmation = accuracy["confirmation"]
print("사용자 답:", confirmation["answers"], "/ 확인 때문에 판정이 바뀐 문장:", confirmation["changed_sentences"])
display(pd.DataFrame(confirmation["accuracy"]))
print("확인으로 점수가 바뀐 연습")
pd.DataFrame(confirmation["changed_take_scores"]).T

사용자 답: {'STT 대로 말함 (발표자가 다른 말을 함)': 3, '대본대로 말함 (음성 인식이 잘못 적음)': 26} / 확인 때문에 판정이 바뀐 문장: 3


,처음 채점,사용자 확인 뒤
대본 문장 정확도,0.992,0.996
Key Point 정확도,0.992,0.996
Key Point 심각한 오판 비율,0.004,0.000
사실 정확도 (보류 제외),0.999,0.999
보류한 사실,18.000,0.000
슬라이드 내용 점수 평균 오차,0.011,0.002


확인으로 점수가 바뀐 연습


,content_coverage (처음),critical_fact_accuracy (처음),script_fidelity (처음),content_coverage (확인 뒤),critical_fact_accuracy (확인 뒤),script_fidelity (확인 뒤)
가상대본1_take5,0.778,0.802,0.647,0.740,0.787,0.645
가상대본1_take6,0.905,0.947,0.903,0.905,0.951,0.903
가상대본1_take9,0.924,1.000,0.965,0.924,1.000,0.965
가상대본2_take4,0.810,0.921,0.916,0.753,0.879,0.914
가상대본2_take5,0.797,0.841,0.588,0.747,0.803,0.587
가상대본2_take6,0.943,0.946,0.918,0.943,0.955,0.918
가상대본2_take9,0.943,0.952,0.958,0.943,0.955,0.958


## 6. 반복 채점 일관성

같은 STT 를 여러 번 채점해 LLM 판정이 채점할 때마다 얼마나 달라지는지 본 결과입니다
(`experiments/06_stability.py` 가 `reports/results/stt_stability.json` 에 남깁니다).
0번은 처음 채점이고, 1번부터는 LLM 의미 평가와 교차 검증을 다시 부른 결과입니다.

| 지표 | 뜻 |
|---|---|
| 대본 문장 판정 일치율 | 모든 채점에서 같은 판정을 받은 문장 비율 (최종) |
| Key Point 판정 일치율 | 모든 채점에서 같은 판정을 받은 Key Point 비율 (LLM 1차 / 최종) |
| 점수 차이 | 같은 슬라이드 · 발표의 점수가 채점마다 달라진 폭 (최대 - 최소, 100점 만점) |
| 채점별 정확도 | 채점마다 정답 라벨과 비교한 정확도 |
| 충돌 조건 점검 (합산) | 5장 점검표를 모든 채점에 대해 합친 것 — 드물게 뜨는 조건은 한 번의 채점으로는 표본이 작음 |

In [14]:
stability = read_result("stt_stability")
print("모델:", stability["model"], "/ 기능 버전:", stability["feature_version"], "/ 채점 횟수:", stability["samples"])
display(pd.Series(stability["stability"], name="값").to_frame())
print("채점별 정확도 (정답 라벨 대비)")
display(result_table("stt_stability", "accuracy_by_run"))
print("충돌 조건 점검 — 채점 합산")
display(result_table("stt_stability", "conflict_check_all_runs"))
print("LLM 확신도별 1차 오답 (채점 합산)")
display(result_table("stt_stability", "confidence_check_all_runs"))
print("채점마다 최종 판정이 달라진 Key Point:", len(stability["unstable_key_points"]), "개")
pd.DataFrame(stability["unstable_key_points"])

모델:

 openai/gpt-5.6-luna / 기능 버전: 1.0 / 채점 횟수: 3


,값
대본 문장 판정 일치율 (최종),0.992
Key Point 판정 일치율 (LLM 1차),0.988
Key Point 판정 일치율 (최종),0.992
슬라이드 내용 점수 차이 평균,0.610
슬라이드 내용 점수 차이 최대,60.000
발표 전체 내용 점수 차이 최대,3.800
발표 전체 수치 점수 차이 최대,4.800


채점별 정확도 (정답 라벨 대비)


,문장 최종,KP LLM 1차,KP 최종,KP 심각한 오판,사실 정확도 (보류 제외),인식 오류 보류율,보류 중 실제 인식 오류
0,0.992,0.967,0.992,0.004,0.999,1.0,0.897
1,0.990,0.971,0.989,0.004,1.000,1.0,0.897
2,0.988,0.967,0.988,0.006,1.000,1.0,0.897


...

,발동,단독,1차 오답,단독 1차 오답,바로잡음,망침
contradicted,125,66,50,0,49,9
low_confidence,12,10,4,2,2,2
judgment_incomplete,0,0,0,0,0,0
evidence_invalid,8,8,0,0,0,0
evidence_unrelated,3,0,0,0,0,0
fact_missing,0,0,0,0,0,0
sound_alike,59,0,50,0,50,9
speaker_changed,3,3,3,3,3,0
approximation,0,0,0,0,0,0
partial_complete,1,1,1,1,0,0


LLM 확신도별 1차 오답 (채점 합산)


,문장,1차 오답,contradicted 제외 문장,contradicted 제외 1차 오답
high,2283,62,2159,13
medium,12,4,11,3


채점마다 최종 판정이 달라진 Key Point: 6 개


,take_id,slide,kp,판정
0,가상대본1_take8,4,KP3,covered / partial / partial
1,가상대본2_take7,8,KP2,covered / covered / contradicted
2,가상대본2_take8,1,KP2,partial / partial / covered
3,가상대본2_take8,1,KP4,covered / covered / partial
4,가상대본2_take8,3,KP4,covered / partial / covered
5,가상대본2_take8,10,KP1,covered / covered / partial
